In [1]:
import pandas as pd
import numpy as np
import json
import re
import time
import math
from pathlib import Path
from collections import Counter
import requests
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import unicodedata

pd.set_option("display.max_colwidth", 120)
pio.templates.default = "plotly_white"

In [4]:
panel = pd.read_csv("interviews.csv")
COMP_COLS = [f"Q_comparaison_{i}" for i in range(1, 7)]
df = panel[["panelist_id"] + COMPCOLS].copy()
print(panel.shape)
df.head(2)

(800, 48)


,panelist_id,Q_comparaison_1,Q_comparaison_2,Q_comparaison_3,Q_comparaison_4,Q_comparaison_5,Q_comparaison_6
0,67b0d135d362f5886c2e5cfe,"J'ai préféré celle de Bouygues. Elle était plus marrante, plus originale. Je l'ai regardée comme un petit sketch, c'...","Celle de Bouygues, sans hésiter. L'idée de la scène de crime pour du wifi, c'est tellement absurde que tu t'en souvi...","Bouygues. Clairement. Pour son concept. Ils ont osé faire un truc complètement différent, une parodie. Ça sort du ca...","Bouygues. Ça m'a fait rire. C'est une émotion simple, mais c'est positif. La pub Orange, elle ne suscite pas vraimen...","C'est marrant parce que même si j'ai préféré la pub Bouygues, je crois que c'est celle d'Orange qui me ferait le plu...","Je dirais Orange. Leur promesse c'est 'la fiabilité', et toute la pub est construite pour te montrer pourquoi c'est ..."
1,67b0d12ed362f5886c2e5b9c,Hmm... difficile. J'ai bien aimé les deux pour des raisons différentes. Mais je vais dire Bouygues. Juste pour l'ori...,"Celle de Bouygues, je pense. L'idée de la 'police du WiFi' c'est un concept fort, facile à retenir et à raconter. La...","Bouygues, sans aucune hésitation. Pour tout le côté parodie de série policière. C'est un vrai parti pris créatif. Or...","Bouygues m'a fait plus rire. Donc si on parle d'émotion forte, le rire, c'est celle-là. Orange, c'est plus une sympa...","Alors là, c'est marrant, mais je dirais peut-être Orange. Même si j'ai préféré la pub Bouygues. Parce que le message...","Je dirais Orange. Leur promesse c'est la fiabilité, et ils le montrent bien avec des exemples où tout le reste échou..."


In [6]:
def build_verbatim(row):
    parts = [str(row[col]) for col in COMP_COLS if pd.notna(row[col])]
    return " ".join(parts)

df["verbatim"] = df.apply(build_verbatim, axis=1)
df[["panelist_id", "verbatim"]].head(3)

,panelist_id,verbatim
0,67b0d135d362f5886c2e5cfe,"J'ai préféré celle de Bouygues. Elle était plus marrante, plus originale. Je l'ai regardée comme un petit sketch, c'..."
1,67b0d12ed362f5886c2e5b9c,Hmm... difficile. J'ai bien aimé les deux pour des raisons différentes. Mais je vais dire Bouygues. Juste pour l'ori...
2,67b0d12bd362f5886c2e5b09,"J'ai trouvé la pub Bouygues très drôle, mais je crois que je préfère celle d'Orange. Elle est plus simple, elle me p..."


In [7]:
SCHEMA_SCORES = {
    "creativity":        "Rate how creative the ad for each brand feels (inventiveness, boldness of concept).",
    "humour":            "Rate how funny or entertaining each brand's ad is.",
    "originality":       "Rate how original, distinctive, and memorable each brand's ad feels.",
    "reliability_trust": "Rate how much each brand's ad inspires reliability, trust, and reassurance.",
    "intent_to_purchase":"Rate how much each brand's ad makes the panelist want to subscribe or switch.",
    "overall":           "Rate the overall preference for each brand's ad, taking everything into account.",
}

BRANDS = ["Bouygues", "Orange"]
ALL_DIMS = list(SCHEMA_SCORES.keys())

DIM_GROUPS = {
    "creative":    ["creativity", "humour", "originality"],
    "commercial":  ["reliability_trust", "intent_to_purchase"],
    "overall":     ["overall"],
}

DIM_LABELS = {
    "creativity":         "Creativity",
    "humour":             "Humour",
    "originality":        "Originality",
    "reliability_trust":  "Reliability / Trust",
    "intent_to_purchase": "Intent to purchase",
    "overall":            "Overall",
}

In [8]:
_schema_lines = "\n".join(
    f"- {dim}: {rule}" for dim, rule in SCHEMA_SCORES.items()
)
_fields_lines = "\n".join(
    f'  "{dim}": {{"Bouygues": <0-10>, "Orange": <0-10>}},'
    for dim in SCHEMA_SCORES
)

SYSTEM_PROMPT = f"""Tu es un annotateur expert en analyse de discours publicitaire.
Tu analyses des verbatims de panlistes ayant regardé deux publicités télévisées :
- Bouygues Telecom : humour absurde, scénario policier décalé
- Orange : message de fiabilité réseau, ton rassurant

Pour chaque verbatim, attribue un score de 0 à 10 à chaque marque sur chaque dimension.
0 = pas du tout / absent, 10 = extrêmement fort / dominant.

DIMENSIONS :
{_schema_lines}

RÈGLES STRICTES :
1. Réponds UNIQUEMENT avec un objet JSON valide, sans texte avant ou après.
2. Chaque score est un entier entre 0 et 10.
3. Ajoute un champ "reasoning" très court.
4. Ajoute un champ "confidence" entre 0.0 et 1.0.

FORMAT DE SORTIE :
{{
{_fields_lines}
  "reasoning": "explication courte",
  "confidence": 0.95
}}"""

FEW_SHOTS = [
    {
        "verbatim": "J'ai préféré celle de Bouygues. Elle était plus drôle, plus originale. Mais pour choisir un opérateur, Orange me rassure davantage sur la fiabilité.",
        "label": {
            "creativity":         {"Bouygues": 8, "Orange": 4},
            "humour":             {"Bouygues": 9, "Orange": 3},
            "originality":        {"Bouygues": 8, "Orange": 3},
            "reliability_trust":  {"Bouygues": 4, "Orange": 9},
            "intent_to_purchase": {"Bouygues": 5, "Orange": 8},
            "overall":            {"Bouygues": 6, "Orange": 7},
        },
    },
    {
        "verbatim": "Bouygues, sans hésiter. Le concept est plus fort, plus drôle, plus original. Et au final la marque me donne aussi confiance.",
        "label": {
            "creativity":         {"Bouygues": 9, "Orange": 3},
            "humour":             {"Bouygues": 9, "Orange": 2},
            "originality":        {"Bouygues": 9, "Orange": 2},
            "reliability_trust":  {"Bouygues": 7, "Orange": 5},
            "intent_to_purchase": {"Bouygues": 8, "Orange": 4},
            "overall":            {"Bouygues": 9, "Orange": 3},
        },
    },
    {
        "verbatim": "Je préfère Orange. C'est moins drôle, mais plus concret, plus crédible, plus rassurant. C'est clairement celle qui me donnerait envie de changer d'opérateur.",
        "label": {
            "creativity":         {"Bouygues": 5, "Orange": 5},
            "humour":             {"Bouygues": 5, "Orange": 3},
            "originality":        {"Bouygues": 5, "Orange": 4},
            "reliability_trust":  {"Bouygues": 3, "Orange": 9},
            "intent_to_purchase": {"Bouygues": 2, "Orange": 9},
            "overall":            {"Bouygues": 3, "Orange": 8},
        },
    },
]

In [9]:
def extract_json(text):
    try:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        return json.loads(m.group()) if m else None
    except Exception:
        return None

CONFIDENCE_THRESHOLD = 0.7

def build_user_prompt(verbatim):
    prompt = "EXEMPLES ANNOTÉS :\n"
    for ex in FEW_SHOTS:
        prompt += f"Verbatim: {ex['verbatim']}\n"
        prompt += f"Annotation: {json.dumps(ex['label'], ensure_ascii=False)}\n---\n"
    prompt += f"À ANNOTER :\n{verbatim}"
    return prompt

def validate_scores(parsed):
    for dim in SCHEMA_SCORES:
        if dim not in parsed:
            parsed[dim] = {"Bouygues": -1, "Orange": -1}
        else:
            for brand in BRANDS:
                val = parsed[dim].get(brand, -1)
                try:
                    parsed[dim][brand] = max(0, min(10, int(val)))
                except (TypeError, ValueError):
                    parsed[dim][brand] = -1
    return parsed

def annotate(verbatim):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_user_prompt(verbatim)},
    ]
    resp = requests.post(
        "http://localhost:11434/api/chat",
        json={"model": "mistral", "messages": messages, "stream": False, "options": {"temperature": 0.1}},
        timeout=120,
    )
    generated = resp.json()["message"]["content"]
    parsed = extract_json(generated)
    if parsed is None:
        out = {dim: {"Bouygues": -1, "Orange": -1} for dim in SCHEMA_SCORES}
        out["reasoning"] = generated[:200]
        out["confidence"] = 0.0
        out["raw_output"] = generated
        return out
    parsed = validate_scores(parsed)
    confidence = float(parsed.get("confidence", 0.0))
    for dim in SCHEMA_SCORES:
        parsed[f"{dim}_flagged"] = confidence < CONFIDENCE_THRESHOLD
    parsed["raw_output"] = generated
    return parsed

In [11]:
test = annotate("J'ai préféré Bouygues pour l'humour et l'originalité, mais Orange me rassure davantage et me donnerait plus envie de m'abonner.")
{k: v for k, v in test.items() if k != "raw_output"}

{'creativity': {'Bouygues': 8, 'Orange': 5},
 'humour': {'Bouygues': 9, 'Orange': 4},
 'originality': {'Bouygues': 8, 'Orange': 3},
 'reliability_trust': {'Bouygues': 6, 'Orange': 8},
 'intent_to_purchase': {'Bouygues': 7, 'Orange': 9},
 'overall': {'Bouygues': 7, 'Orange': 8},
 'reasoning': "Le scénario décalé et l'humour absurde de Bouygues ont été appréciés pour leur originalité et leur amusement. Cependant, le message de fiabilité réseau et le ton rassurant d'Orange ont inspiré plus de confiance et de volonté d'abonnement.",
 'confidence': 0.95,
 'creativity_flagged': False,
 'humour_flagged': False,
 'originality_flagged': False,
 'reliability_trust_flagged': False,
 'intent_to_purchase_flagged': False,
 'overall_flagged': False}

In [13]:
PILOT_N = 30
pilot_df = df.sample(n=PILOT_N, random_state=42).copy().reset_index(drop=True)

pilot_results = []
for i, row in pilot_df.iterrows():
    t0 = time.time()
    result = annotate(row["verbatim"])
    result["panelist_id"] = row["panelist_id"]
    pilot_results.append(result)
    elapsed = time.time() - t0
    print(
        f"{i+1:3d}/{PILOT_N} {elapsed:.1f}s | "
        + " ".join(f"{d}={result[d]['Bouygues']}v{result[d]['Orange']}" for d in ALL_DIMS)
        + f" | conf={result.get('confidence','?')}"
    )

pilot_results = pd.DataFrame(pilot_results)
pilot_results.to_csv("pilot_annotations_scores.csv", index=False)
pilot_results.head()

  1/30 10.5s | creativity=6v7 humour=5v8 originality=4v7 reliability_trust=3v9 intent_to_purchase=2v10 overall=4v9 | conf=0.95
  2/30 10.3s | creativity=7v6 humour=5v8 originality=6v9 reliability_trust=4v10 intent_to_purchase=2v10 overall=4v9 | conf=0.95
  3/30 10.0s | creativity=9v5 humour=7v3 originality=10v4 reliability_trust=4v9 intent_to_purchase=5v8 overall=6v8 | conf=0.95
  4/30 8.9s | creativity=8v7 humour=5v9 originality=10v6 reliability_trust=3v8 intent_to_purchase=4v9 overall=5v8 | conf=0.95
  5/30 9.9s | creativity=7v6 humour=6v5 originality=9v6 reliability_trust=4v8 intent_to_purchase=3v7 overall=5v8 | conf=0.95
  6/30 10.6s | creativity=9v5 humour=8v4 originality=10v6 reliability_trust=7v9 intent_to_purchase=5v8 overall=6v8 | conf=0.95
  7/30 9.9s | creativity=9v7 humour=8v8 originality=10v6 reliability_trust=4v9 intent_to_purchase=5v8 overall=6v9 | conf=0.95
  8/30 9.9s | creativity=9v6 humour=7v5 originality=10v4 reliability_trust=3v9 intent_to_purchase=5v8 overall=6v8 

,creativity,humour,originality,reliability_trust,intent_to_purchase,overall,reasoning,confidence,creativity_flagged,humour_flagged,originality_flagged,reliability_trust_flagged,intent_to_purchase_flagged,overall_flagged,raw_output,panelist_id
0,"{'Bouygues': 6, 'Orange': 7}","{'Bouygues': 5, 'Orange': 8}","{'Bouygues': 4, 'Orange': 7}","{'Bouygues': 3, 'Orange': 9}","{'Bouygues': 2, 'Orange': 10}","{'Bouygues': 4, 'Orange': 9}",The Orange ad was simpler and more relatable to everyday life. It focused on the benefits of the service and inspire...,0.95,False,False,False,False,False,False,"{\n ""creativity"": {""Bouygues"": 6, ""Orange"": 7},\n ""humour"": {""Bouygues"": 5, ""Orange"": 8},\n ""originality"": {""Bou...",67b0d131d362f5886c2e5c61
1,"{'Bouygues': 7, 'Orange': 6}","{'Bouygues': 5, 'Orange': 8}","{'Bouygues': 6, 'Orange': 9}","{'Bouygues': 4, 'Orange': 10}","{'Bouygues': 2, 'Orange': 10}","{'Bouygues': 4, 'Orange': 9}","The Orange ad was preferred due to its simplicity, clarity, and human touch. The Bouygues ad was too complicated and...",0.95,False,False,False,False,False,False,"{\n ""creativity"": {""Bouygues"": 7, ""Orange"": 6},\n ""humour"": {""Bouygues"": 5, ""Orange"": 8},\n ""originality"": {""Bou...",67b0d12fd362f5886c2e5bdd
2,"{'Bouygues': 9, 'Orange': 5}","{'Bouygues': 7, 'Orange': 3}","{'Bouygues': 10, 'Orange': 4}","{'Bouygues': 4, 'Orange': 9}","{'Bouygues': 5, 'Orange': 8}","{'Bouygues': 6, 'Orange': 8}",The Orange ad speaks more directly to the panelist's daily concerns and is more persuasive. It focuses on the benefi...,0.95,False,False,False,False,False,False,"{\n ""creativity"": {""Bouygues"": 9, ""Orange"": 5},\n ""humour"": {""Bouygues"": 7, ""Orange"": 3},\n ""originality"": {""Bou...",67b0d125d362f5886c2e59f3
3,"{'Bouygues': 8, 'Orange': 7}","{'Bouygues': 5, 'Orange': 9}","{'Bouygues': 10, 'Orange': 6}","{'Bouygues': 3, 'Orange': 8}","{'Bouygues': 4, 'Orange': 9}","{'Bouygues': 5, 'Orange': 8}","The Orange ad was more relatable and humorous, while the Bouygues ad was original but complex and less emotionally e...",0.95,False,False,False,False,False,False,"{\n ""creativity"": {""Bouygues"": 8, ""Orange"": 7},\n ""humour"": {""Bouygues"": 5, ""Orange"": 9},\n ""originality"": {""Bou...",67b0d12dd362f5886c2e5b91
4,"{'Bouygues': 7, 'Orange': 6}","{'Bouygues': 6, 'Orange': 5}","{'Bouygues': 9, 'Orange': 6}","{'Bouygues': 4, 'Orange': 8}","{'Bouygues': 3, 'Orange': 7}","{'Bouygues': 5, 'Orange': 8}","The Orange ad was perceived as simpler, clearer and warmer, creating a stronger emotional connection. The Bouygues a...",0.95,False,False,False,False,False,False,"{\n ""creativity"": {""Bouygues"": 7, ""Orange"": 6},\n ""humour"": {""Bouygues"": 6, ""Orange"": 5},\n ""originality"": {""Bou...",67b0d125d362f5886c2e59ff


In [15]:
CHECKPOINT_EVERY = 50
checkpoint_path = "annotations_scores_checkpoint.csv"
all_results = []
run_start = time.time()
total = len(df)

for i, row in df.iterrows():
    t0 = time.time()
    result = annotate(row["verbatim"])
    result["panelist_id"] = row["panelist_id"]
    all_results.append(result)
    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(all_results).to_csv(checkpoint_path, index=False)
        done = i + 1
        avg = (time.time() - run_start) / done
        eta = avg * (total - done) / 60
        print(f"{done}/{total} | checkpoint saved | {avg:.1f}s/sample | ETA {eta:.0f} min")

annotations = pd.DataFrame(all_results)
annotations.to_csv("preference_annotations_scores.csv", index=False)
annotations.head()

50/800 | checkpoint saved | 10.0s/sample | ETA 124 min
100/800 | checkpoint saved | 9.9s/sample | ETA 115 min
150/800 | checkpoint saved | 18.0s/sample | ETA 195 min
200/800 | checkpoint saved | 16.0s/sample | ETA 160 min
250/800 | checkpoint saved | 15.7s/sample | ETA 144 min
300/800 | checkpoint saved | 15.9s/sample | ETA 132 min
350/800 | checkpoint saved | 15.0s/sample | ETA 112 min
400/800 | checkpoint saved | 14.4s/sample | ETA 96 min
450/800 | checkpoint saved | 13.9s/sample | ETA 81 min
500/800 | checkpoint saved | 13.5s/sample | ETA 67 min
550/800 | checkpoint saved | 13.2s/sample | ETA 55 min
600/800 | checkpoint saved | 12.9s/sample | ETA 43 min
650/800 | checkpoint saved | 12.6s/sample | ETA 32 min
700/800 | checkpoint saved | 17.8s/sample | ETA 30 min
750/800 | checkpoint saved | 17.9s/sample | ETA 15 min
800/800 | checkpoint saved | 17.5s/sample | ETA 0 min


,creativity,humour,originality,reliability_trust,intent_to_purchase,overall,reasoning,confidence,creativity_flagged,humour_flagged,originality_flagged,reliability_trust_flagged,intent_to_purchase_flagged,overall_flagged,raw_output,panelist_id
0,"{'Bouygues': 9, 'Orange': 5}","{'Bouygues': 8, 'Orange': 4}","{'Bouygues': 9, 'Orange': 6}","{'Bouygues': 7, 'Orange': 9}","{'Bouygues': 5, 'Orange': 8}","{'Bouygues': 7, 'Orange': 7}","The Bouygues ad was considered more creative, humorous, and original due to its absurd humor and unique concept. The...",0.95,False,False,False,False,False,False,"{\n ""creativity"": {""Bouygues"": 9, ""Orange"": 5},\n ""humour"": {""Bouygues"": 8, ""Orange"": 4},\n ""originality"": {""Bou...",67b0d135d362f5886c2e5cfe
1,"{'Bouygues': 9, 'Orange': 5}","{'Bouygues': 8, 'Orange': 4}","{'Bouygues': 10, 'Orange': 6}","{'Bouygues': 7, 'Orange': 9}","{'Bouygues': 5, 'Orange': 8}","{'Bouygues': 7, 'Orange': 7}",The 'WiFi police' concept in Bouygues ad is original and memorable. The Orange ad effectively communicates reliabili...,0.95,False,False,False,False,False,False,"{\n ""creativity"": {""Bouygues"": 9, ""Orange"": 5},\n ""humour"": {""Bouygues"": 8, ""Orange"": 4},\n ""originality"": {""Bou...",67b0d12ed362f5886c2e5b9c
2,"{'Bouygues': 9, 'Orange': 5}","{'Bouygues': 8, 'Orange': 4}","{'Bouygues': 10, 'Orange': 6}","{'Bouygues': 5, 'Orange': 9}","{'Bouygues': 4, 'Orange': 8}","{'Bouygues': 7, 'Orange': 8}","The absurd detective scenario in Bouygues ad is highly original and humorous, but the Orange ad's focus on everyday ...",0.95,False,False,False,False,False,False,"{\n ""creativity"": {""Bouygues"": 9, ""Orange"": 5},\n ""humour"": {""Bouygues"": 8, ""Orange"": 4},\n ""originality"": {""Bou...",67b0d12bd362f5886c2e5b09
3,"{'Bouygues': 7, 'Orange': 5}","{'Bouygues': 6, 'Orange': 3}","{'Bouygues': 9, 'Orange': 4}","{'Bouygues': 5, 'Orange': 8}","{'Bouygues': 3, 'Orange': 9}","{'Bouygues': 4, 'Orange': 8}","The Orange ad is simpler and more direct, with a clear message of reliability. The Bouygues ad is more creative and ...",0.95,False,False,False,False,False,False,"{\n ""creativity"": {""Bouygues"": 7, ""Orange"": 5},\n ""humour"": {""Bouygues"": 6, ""Orange"": 3},\n ""originality"": {""Bou...",67b0d127d362f5886c2e5a4a
4,"{'Bouygues': 9, 'Orange': 5}","{'Bouygues': 8, 'Orange': 4}","{'Bouygues': 10, 'Orange': 6}","{'Bouygues': 7, 'Orange': 9}","{'Bouygues': 5, 'Orange': 8}","{'Bouygues': 7, 'Orange': 7}","The Bouygues ad stands out for its originality and creativity, with a unique concept that is memorable. The humor an...",0.95,False,False,False,False,False,False,"{\n ""creativity"": {""Bouygues"": 9, ""Orange"": 5},\n ""humour"": {""Bouygues"": 8, ""Orange"": 4},\n ""originality"": {""Bou...",67b0d12cd362f5886c2e5b4a


In [17]:
final = pd.read_csv("preference_annotations_scores.csv")

for dim in ALL_DIMS:
    for brand in BRANDS:
        col = f"{dim}_{brand.lower()}"
        if isinstance(final[dim].iloc[0], str):
            final[col] = final[dim].apply(
                lambda x: json.loads(x.replace("'", '"')).get(brand, np.nan)
                if pd.notna(x) else np.nan
            )
        else:
            final[col] = final[dim].apply(lambda x: x.get(brand, np.nan) if isinstance(x, dict) else np.nan)

score_cols = [f"{dim}_{b.lower()}" for dim in ALL_DIMS for b in BRANDS]
cols_to_merge = ["panelist_id"] + score_cols + ["confidence", "reasoning"]
panel_merged = panel.merge(final[cols_to_merge], on="panelist_id", how="left")
panel_merged.to_csv("interviews_with_scores.csv", index=False)
print(panel_merged.shape)
panel_merged[["panelist_id"] + score_cols[:6]].head()

(800, 62)


,panelist_id,creativity_bouygues,creativity_orange,humour_bouygues,humour_orange,originality_bouygues,originality_orange
0,67b0d135d362f5886c2e5cfe,9,5,8,4,9,6
1,67b0d12ed362f5886c2e5b9c,9,5,8,4,10,6
2,67b0d12bd362f5886c2e5b09,9,5,8,4,10,6
3,67b0d127d362f5886c2e5a4a,7,5,6,3,9,4
4,67b0d12cd362f5886c2e5b4a,9,5,8,4,10,6


In [18]:
summary = pd.DataFrame({
    "Dimension": [DIM_LABELS[d] for d in ALL_DIMS],
    "Bouygues":  [round(final[f"{d}_bouygues"].mean(), 2) for d in ALL_DIMS],
    "Orange":    [round(final[f"{d}_orange"].mean(), 2) for d in ALL_DIMS],
}).set_index("Dimension")

summary.to_csv("score_summary.csv")
summary

,Bouygues,Orange
Dimension,,
Creativity,8.19,5.46
Humour,7.13,4.96
Originality,9.26,5.24
Reliability / Trust,4.64,8.66
Intent to purchase,5.19,7.98
Overall,6.25,7.57


In [19]:
BRAND_COLORS = {"Bouygues": "#0055A4", "Orange": "#FF6600"}
MARGINS = dict(t=80, b=120, l=60, r=20)
LEGEND_BELOW = dict(orientation="h", yanchor="top", y=-0.18, xanchor="center", x=0.5)

DEMOGRAPHICS = {
    "gender":              "Gender",
    "agegroup":            "Age",
    "csp":                 "CSP",
    "income_level":        "Income level",
    "education":           "Education",
    "location.citysize":   "City size",
}

available_demographics = {k: v for k, v in DEMOGRAPHICS.items() if k in panel_merged.columns}

In [20]:
if "age" in panel_merged.columns:
    panel_merged["agegroup"] = pd.cut(
        panel_merged["age"],
        bins=[17, 24, 34, 44, 54, 64, 120],
        labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"],
    )

In [21]:
def prep_demo_col(df, col, min_count=10, max_levels=12):
    out = df.copy()
    out = out[out[col].notna()].copy()
    out["demo"] = out[col].astype(str)
    out = out[out["demo"] != "nan"].copy()
    counts = out["demo"].value_counts()
    keep = counts[counts >= min_count].index.tolist()
    if len(keep) == 0:
        keep = counts.index.tolist()
    out["demo"] = np.where(out["demo"].isin(keep), out["demo"], "Other")
    counts2 = out["demo"].value_counts()
    top_levels = counts2.index.tolist()[:max_levels]
    out = out[out["demo"].isin(top_levels)].copy()
    return out

In [22]:
def make_score_facet_plot(df, dim, demo_col, demo_label):
    plot_df = prep_demo_col(df, demo_col)
    score_cols_dim = [f"{dim}_{b.lower()}" for b in BRANDS]
    long = plot_df[["demo"] + score_cols_dim].melt(
        id_vars="demo",
        value_vars=score_cols_dim,
        var_name="brand_col",
        value_name="score",
    )
    long["brand"] = long["brand_col"].str.replace(f"{dim}_", "", regex=False).str.capitalize()
    long["brand"] = long["brand"].replace({"Bouygues": "Bouygues", "Orange": "Orange"})

    grouped = (
        long.groupby(["demo", "brand"])["score"]
        .agg(mean_score="mean", sem=lambda x: x.sem())
        .reset_index()
    )
    grouped.rename(columns={"demo": demo_label}, inplace=True)

    n_levels = grouped[demo_label].nunique()
    n_cols = min(3, max(1, n_levels))
    n_rows = math.ceil(n_levels / n_cols)

    fig = px.bar(
        grouped,
        x="brand",
        y="mean_score",
        color="brand",
        facet_col=demo_label,
        facet_col_wrap=n_cols,
        error_y="sem",
        color_discrete_map=BRAND_COLORS,
        title=f"{DIM_LABELS[dim]} by {demo_label}",
        range_y=[0, 10],
    )
    fig.update_layout(
        showlegend=False,
        margin=dict(t=80, b=50, l=50, r=20),
        height=max(450, 260 * n_rows),
    )
    fig.update_yaxes(title_text="Mean score (0–10)", matches=None)
    fig.update_xaxes(title_text="")
    return fig

In [23]:
fig = go.Figure()
for brand in BRANDS:
    fig.add_trace(go.Bar(
        name=brand,
        x=[DIM_LABELS[d] for d in ALL_DIMS],
        y=[round(final[f"{d}_{brand.lower()}"].mean(), 2) for d in ALL_DIMS],
        marker_color=BRAND_COLORS[brand],
    ))

fig.update_layout(
    barmode="group",
    title=dict(text="Mean scores across all dimensions", x=0, xanchor="left"),
    legend=LEGEND_BELOW,
    margin=MARGINS,
    yaxis=dict(title="Mean score (0–10)", range=[0, 10]),
    xaxis=dict(title=""),
)
fig.show()

In [24]:
for dim in ALL_DIMS:
    for demo_col, demo_label in available_demographics.items():
        fig = make_score_facet_plot(panel_merged, dim, demo_col, demo_label)
        fig.show()

In [25]:
def slugify(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"[^\w]+", "_", s).strip("_").lower()

output_dir = Path("score_facet_charts")
output_dir.mkdir(exist_ok=True)
saved_files = []

for dim in ALL_DIMS:
    for demo_col, demo_label in available_demographics.items():
        fig = make_score_facet_plot(panel_merged, dim, demo_col, demo_label)
        plot_df = prep_demo_col(panel_merged, demo_col)
        n_levels = plot_df["demo"].nunique() if len(plot_df) else 1
        n_cols = min(3, max(1, n_levels))
        n_rows = max(1, math.ceil(n_levels / n_cols))
        fig.update_layout(
            width=2400,
            height=max(1000, 420 * n_rows),
            margin=dict(t=100, b=80, l=70, r=40),
            title_x=0.01,
            font=dict(size=16),
        )
        fig.update_annotations(font_size=15)
        out = output_dir / f"{slugify(dim)}_by_{slugify(demo_col)}.png"
        fig.write_image(str(out), width=2400, height=max(1000, 420 * n_rows), scale=2)
        saved_files.append(str(out))

len(saved_files)

24